In [7]:
import pandas as pd
import os 
from sqlalchemy import create_engine

# 1. Re-establish the manager connection
USER = 'postgres'
PASSWORD = 'history123' # Replace with your actual password
HOST = 'localhost'
PORT = '5432'
DB_NAME = 'Inventory_db'   # Your database name

connection_string = f'postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}'
engine = create_engine(connection_string)



In [8]:
df = pd.read_sql_query("SELECT * FROM sales LIMIT 5;", engine)
print("Connected successfully! Here is a preview of your data:")
df

Connected successfully! Here is a preview of your data:


,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-01,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
1,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,2,32.98,16.49,2024-01-02,750.0,1,1.57,12546,JIM BEAM BRANDS COMPANY
2,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-03,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
3,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,14.49,14.49,2024-01-08,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
4,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,2,69.98,34.99,2024-01-09,375.0,1,0.79,12546,JIM BEAM BRANDS COMPANY


In [9]:
df = pd.read_sql_query('SELECT * FROM vendor_invoice',engine)
df.head()

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,None
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,None
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,None


In [17]:
import pandas as pd
import os
import io
import logging

os.makedirs("Logs", exist_ok =True)

#config the loging 
logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s [%(levelname)s] %(message)s", #takes time from computer time automatically
    handlers=[
        logging.FileHandler(
            "logs/pipeline.log"
        ),
        logging.StreamHandler(),
    ],
)


def ingest_db(df, table_name,  engine):
    df.columns = df.columns.str.strip()
    logging.info(f"starting database creation for table: '{table_name}'")

    #using native postgres command to better optimize the code for larger datasets
    try:
        df.head(0).to_sql(table_name, con = engine, if_exists = 'replace' , index = False)
        raw_conn = engine.raw_connection()
        with raw_conn.cursor() as cursor:
            output = io.StringIO()
            df.to_csv(output, sep=',',header = False, index = False)
            output.seek(0)
            
            sql = f"COPY {table_name} FROM STDIN WITH CSV DELIMITER ','"
            cursor.copy_expert(sql, output)
            raw_conn.commit()
            print(f" {table_name} imported successfully")
            logging.info( f"success: '{table_name}' imported ")
    except Exception as e:
        print(f"ERROR importing {table_name}: {e}")
        logging.error( f"failed: '{table_name}' imported ")

    finally:
        raw_conn.close()

    #Run your file through loop
def load_raw_data():
    for file in os.listdir('.'):
        if '.csv' in file:
            table_name = file[:-4]
            try: 
                logging.info(f"reading file '{file}'")
                df = pd.read_csv(file)
                ingest_db(df, file[:-4], engine)
            except Exception as e:
                logging.error(f" failed to read '{file}':{e}")
logging.info("data ingestion pipeline finished")
if __name__ == '__main__':
    load_raw_data()

2026-06-27 22:48:16,816 [INFO] data ingestion pipeline finished
2026-06-27 22:48:16,817 [INFO] reading file 'begin_inventory.csv'
2026-06-27 22:48:17,052 [INFO] starting database creation for table: 'begin_inventory'
2026-06-27 22:48:17,821 [INFO] success: 'begin_inventory' imported 
2026-06-27 22:48:17,826 [INFO] reading file 'end_inventory.csv'


 begin_inventory imported successfully


2026-06-27 22:48:18,084 [INFO] starting database creation for table: 'end_inventory'
2026-06-27 22:48:18,975 [INFO] success: 'end_inventory' imported 
2026-06-27 22:48:18,981 [INFO] reading file 'purchases.csv'


 end_inventory imported successfully


2026-06-27 22:48:23,381 [INFO] starting database creation for table: 'purchases'
2026-06-27 22:48:44,197 [INFO] success: 'purchases' imported 
2026-06-27 22:48:44,394 [INFO] reading file 'purchase_prices.csv'


...

 purchase_prices imported successfully


2026-06-27 22:49:08,416 [INFO] starting database creation for table: 'sales'
2026-06-27 22:51:00,643 [INFO] success: 'sales' imported 


 sales imported successfully


2026-06-27 22:51:01,385 [INFO] reading file 'vendor_invoice.csv'
2026-06-27 22:51:02,639 [INFO] starting database creation for table: 'vendor_invoice'
2026-06-27 22:51:02,837 [INFO] success: 'vendor_invoice' imported 


 vendor_invoice imported successfully
